# 01 · Lines

Every transition SkewerJAX can compute is one row of `skewerjax/core/lines_table.dat`: rest wavelength λ0 [Å],
oscillator strength f, Einstein coefficient A21 [Hz] and the mass of the absorbing atom [amu] (the Lyman series,
D I, He, Be, B, C, N, O, Ne, Mg, Si and Fe lines). `Line` is one transition; `LineBundle` holds the transitions of
one calculation as per-line arrays (what the optical depth needs, 06) and their velocity offsets Δv_j from a reference
line (what puts every line on one velocity axis, 07). Runtime: seconds.

**Requires:** nothing (the atomic data of `lines_table.dat`).  
**Independent of:** cosmology, skewers, fields: none of them is built here.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, use_backend, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM,
                                      CMAPS, LABELS, FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # dark style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

In [ ]:
plot_dag(('lines',)); plt.show()

## 1 · The line table

In [ ]:
from skewerjax.core import LINES, Line, LineBundle, get_line, get_mass_amu, phys
from skewerjax.utils.plotting import plot_line_table
species = sorted({l.species for l in LINES.values()})
print(f"{len(LINES)} transitions of {len(species)} species in lines_table.dat:", ", ".join(species))
print("Lyman series:", sum(l.species == "HI" and l.A21_Hz > 0 for l in LINES.values()), "| lines with A21 = 0 (purely thermal test lines):",
      [l.id for l in LINES.values() if l.A21_Hz == 0])

## 2 · One transition: `Line`

A frozen record with the columns of the table (`id`, `species`, `name`, `lambda0_A`, `f_osc`, `A21_Hz`, `mass_amu`) and
one property, `A21_GHz`. The two widths of the Voigt profile follow from it (review notes, Eq. line_profile;
`core.optical_depth.voigt_widths_GHz`): the Lorentzian half-width γ = A21/4π and the Gaussian
σ = b/(√2 λ0) with b = √(2 k_B T/m).

In [ ]:
lya = get_line("HI-LyA")
print(lya)
gamma_GHz = lya.A21_GHz / (4 * np.pi)
b_kms = np.sqrt(2 * phys.kB * 1e4 / lya.mass_amu) / 1e3                      # phys.kB = k_B / m_u [m^2 s^-2 K^-1]
print(f"lambda0 = {lya.lambda0_A} A | f = {lya.f_osc} | A21 = {lya.A21_Hz:.4e} Hz = {lya.A21_GHz:.4f} GHz | m = {lya.mass_amu} amu")
print(f"gamma = A21 / 4 pi = {gamma_GHz:.4f} GHz, i.e. lambda0 gamma = {lya.lambda0_A * 1e-10 * gamma_GHz * 1e9 / 1e3:.2e} km/s; "
      f"thermal b(10^4 K) = {b_kms:.2f} km/s")

## 3 · The transitions of one calculation: `LineBundle`

`LineBundle(line_ids, ref_line_id)` keeps the `Line` records (`lines`, `ref_line`) and exposes, per line, the arrays
`lambda0_A`, `f_osc`, `A21_GHz`, `mass_amu` and `delta_v` = c ln(λ_ref/λ_j) [km/s]. The species bookkeeping
(`species_keys`, `indices_for_species`, `lines_for_species`) maps lines to the gas fields of each species: lines of
one species share its fields. The bundle is a pytree whose leaves are the four arrays.

In [ ]:
lines = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206", "SiII-1260", "SiII-1193", "CIV-1548", "CIV-1551", "MgI-2852"], ref_line_id="HI-LyA")
print(f"n_lines = {lines.n_lines} | species_keys = {lines.species_keys} | n_species = {lines.n_species} | "
      f"ref_line_id = {lines.ref_line_id} ({lines.ref_lambda0_A} A), index {lines.ref_index()}")
for name in ("lambda0_A", "f_osc", "A21_GHz", "mass_amu", "delta_v"):
    print(f"  {name:10s}", np.array2string(np.asarray(getattr(lines, name)), precision=4, max_line_width=150))
print("indices_for_species('SiII') =", lines.indices_for_species("SiII"), "| lines_for_species('CIV') =",
      [l.id for l in lines.lines_for_species("CIV")], "| delta_v_for_line('SiIII-1206') =", f"{lines.delta_v_for_line('SiIII-1206'):.1f} km/s")
print("get_mass_amu(['H', 'C', 'Si']) =", get_mass_amu(["H", "C", "Si"]))
leaves = jax.tree_util.tree_leaves(lines)
print(f"pytree: {len(leaves)} leaves of shape {leaves[0].shape}; jit(lambda b: b.delta_v)(lines) = "
      f"{np.round(np.asarray(jax.jit(lambda b: b.delta_v)(lines))[:3], 1)} ...")

In [ ]:
fig, axs = plot_line_table(lines)
add_panel_letters(axs)
fig.suptitle("The lines of a bundle: strength (a), where they land on the Lyα frame (b), and their two widths (c)", fontsize=21)
plt.show()

*Reading the figure.* (a) The oscillator strength sets how much column a line needs to absorb. (b) Δv_j is the
velocity at which line j's absorption appears on the Lyα frame (negative: bluer than Lyα): Si III 1206 lands close
to Lyα and inside the forest, which is why it correlates with it (07); lines with large |Δv| fall outside the forest of
the same gas. (c) The natural (Lorentzian) widths are orders of magnitude below the thermal widths at 10⁴ K, so line
cores are Gaussian and the Lorentzian matters only in the damping wings of strong absorbers.